# Milan, Sentinel-2 stack (2 dates), CLMS — split (train/test), incremental-dates sweep

One step of the user's incremental-dates test: 2 dates
(20 bands) in the build-up order Oct → Mar → Jun → Aug → Dec → Apr
(not chronological -- the user's own choice). Dates used here: 2018-10-24, 2018-03-23.

Uses the SAME train/test partition as the original 4-date stack (by point
identity, from `./output/milan/clms/stack`) at every step of this sweep, so the holdout
numbers across all six steps are directly comparable -- the only thing that
changes between them is the composite. The 4-date and 5-date steps of this
sweep are not built as separate notebooks: `build_composite`'s stack method
sorts dates internally, so a composite is fully determined by its set of
dates, and those two sets are exactly the existing 4-date stack and the
5-date stack test already run elsewhere -- their results are reused directly
rather than recomputed.

## Cell 1 · Imports & configuration

In [1]:
import ee
from gee_init import init_gee

init_gee()

import json
import os
import time
import warnings

import geemap
import geopandas as gpd
import joblib
import numpy as np
import pandas as pd
from googleapiclient.discovery import build as gbuild
from googleapiclient.http import MediaIoBaseDownload
from scipy.spatial import cKDTree
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.model_selection import RandomizedSearchCV

from s2_utils import build_composite, water_mask

warnings.filterwarnings('ignore')

RANDOM_STATE = 42
N_JOBS       = -1
N_FOLDS      = 5
BLOCK_SIZES  = [500, 1000, 2000]
N_ITER_RF    = 40
LABEL_COL    = 'IMD'

TARGET      = 'CLMS'
SRC_DIR     = './output/milan/clms/stack'
STACK_DATES = ['2018-10-24', '2018-03-23']
N_DATES     = 2

OUTPUT_DIR = './output/milan/clms/stackN2_split'
os.makedirs(OUTPUT_DIR, exist_ok=True)
DRIVE_FOLDER = 'GEE_exports'

print(f'Milan, S2 stack ({N_DATES} dates), {TARGET} -- split (train/test)')
print(f'dates: {STACK_DATES}')
print(f'source points: {SRC_DIR}  ->  output: {OUTPUT_DIR}')

*** Earth Engine *** Share your feedback by taking our Annual Developer Satisfaction Survey: https://google.qualtrics.com/jfe/form/SV_9oS0DRcPvElRMNw?source=python


GEE initialised (project: imd-vietnam).


Milan, S2 stack (2 dates), CLMS -- split (train/test)
dates: ['2018-10-24', '2018-03-23']
source points: ./output/milan/clms/stack  ->  output: ./output/milan/clms/stackN2_split


---
## Cell 2 · AOI, composite, extract at the original stack's point locations

In [2]:
from shapely.geometry import mapping

aoi_gdf  = gpd.read_file('./data/aoi_milan/milano_aoi.gpkg').to_crs('EPSG:4326')
aoi_geom = ee.Geometry(mapping(aoi_gdf.geometry.union_all()))
print(f'Milan AOI area: {aoi_geom.area(1).getInfo() / 1e6:,.0f} km2')

non_water_mask = water_mask(aoi_geom)

SELECTED_DATES = STACK_DATES
pred_source_img, s2_proj, band_names = build_composite(
    'COPERNICUS/S2_SR_HARMONIZED', SELECTED_DATES, aoi_geom,
    non_water=non_water_mask, method='stack')
print(f'S2 stack bands: {len(band_names)}  dates: {len(SELECTED_DATES)}')

# Reuse the ORIGINAL 4-date stack's own point locations + labels + train/test
# partition (geometry and label only -- NOT its 40 bands, since this is a
# fresh composite). Same partition at every step in this date-count sweep,
# so the only variable that changes between runs is the composite itself.
train_src = gpd.read_file(f'{SRC_DIR}/spatial_train_pts.gpkg')
test_src  = gpd.read_file(f'{SRC_DIR}/spatial_test_pts.gpkg')
train_src['split'] = 'train'
test_src['split']  = 'test'
pts = pd.concat([train_src[['geometry', LABEL_COL, 'split']],
                  test_src[['geometry', LABEL_COL, 'split']]], ignore_index=True)
pts = gpd.GeoDataFrame(pts, geometry='geometry', crs=train_src.crs).reset_index(drop=True)
print(f'Reusing {len(train_src)} train + {len(test_src)} test = {len(pts)} point locations from the 4-date stack')

fc_pts = ee.FeatureCollection([
    ee.Feature(ee.Geometry.Point([row.geometry.x, row.geometry.y]),
               {LABEL_COL: float(row[LABEL_COL]), 'split': row['split'], 'pid': int(i)})
    for i, row in pts.iterrows()
])
sampled = pred_source_img.sampleRegions(
    collection=fc_pts, properties=[LABEL_COL, 'split', 'pid'],
    scale=10, projection=s2_proj, tileScale=4, geometries=True)

t0 = time.time()
gdf_all = geemap.ee_to_gdf(sampled)
print(f'extracted {len(gdf_all)} / {len(pts)} points in {time.time() - t0:.1f}s')
n_before = len(gdf_all)
gdf_all = gdf_all.dropna(subset=band_names).reset_index(drop=True)
print(f'after NaN drop: {len(gdf_all)} ({100 * len(gdf_all) / n_before:.1f}% retained)')

gdf_all.to_file(f'{OUTPUT_DIR}/samples_Milan_stackN_all.gpkg', driver='GPKG')

Milan AOI area: 4,455 km2
S2 stack bands: 20  dates: 2
Reusing 2449 train + 1014 test = 3463 point locations from the 4-date stack


extracted 3449 / 3463 points in 134.8s
after NaN drop: 3449 (100.0% retained)


---
## Cell 3 · Spatial block helper functions

In [3]:
def assign_blocks(coords, block_size):
    bx = (coords[:, 0] // block_size).astype(int)
    by = (coords[:, 1] // block_size).astype(int)
    return np.array([f'{x}_{y}' for x, y in zip(bx, by)])

def blocks_to_folds(block_ids, n_folds, seed):
    rng    = np.random.RandomState(seed)
    unique = rng.permutation(np.unique(block_ids))
    bmap   = {b: i % n_folds for i, b in enumerate(unique)}
    return np.array([bmap[b] for b in block_ids])

def apply_buffer(train_coords, test_coords, buffer_m):
    if len(train_coords) == 0 or buffer_m == 0:
        return np.ones(len(test_coords), dtype=bool)
    dists, _ = cKDTree(train_coords).query(test_coords, k=1)
    return dists > buffer_m

def make_spatial_splits(point_folds, n_folds):
    all_idx = np.arange(len(point_folds))
    return [(all_idx[point_folds != f], all_idx[point_folds == f])
            for f in range(n_folds)]

print('Spatial block functions defined.')

Spatial block functions defined.


---
## Cell 3b · Split back into the 4-date stack's own train/test partition

In [4]:
train_gdf = gdf_all[gdf_all['split'] == 'train'].reset_index(drop=True)
test_gdf  = gdf_all[gdf_all['split'] == 'test'].reset_index(drop=True)
print(f'train={len(train_gdf)}  test={len(test_gdf)}  (4-date stack had '
      f'{len(train_src)}/{len(test_src)} before this composite\'s own NaN drop)')

X_train = train_gdf[band_names].values.astype(np.float32)
y_train = train_gdf[LABEL_COL].values.astype(np.float32)
X_test  = test_gdf[band_names].values.astype(np.float32)
y_test  = test_gdf[LABEL_COL].values.astype(np.float32)

train_utm = train_gdf.to_crs('EPSG:32632')
coords_train = np.column_stack([train_utm.geometry.x, train_utm.geometry.y])

train=2440  test=1009  (4-date stack had 2449/1014 before this composite's own NaN drop)


---
## Cell 4 · RF hyperparameter tuning, 5-fold spatial-block CV, on TRAIN only

In [5]:
PARAM_SPACE = {
    'n_estimators':      [100, 200, 300, 500],
    'max_depth':         [None, 10, 20, 30],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf':  [1, 2, 4],
    'max_features':      ['sqrt', 'log2', 0.3, 0.5],
    'max_samples':       [None, 0.7, 0.8, 0.9],
}

best_rmse_tune, best_search, best_block = np.inf, None, None
for bs in BLOCK_SIZES:
    bids = assign_blocks(coords_train, bs)
    pf = blocks_to_folds(bids, N_FOLDS, RANDOM_STATE)
    cv_splits = make_spatial_splits(pf, N_FOLDS)
    t0 = time.time()
    search = RandomizedSearchCV(
        estimator=RandomForestRegressor(random_state=RANDOM_STATE, n_jobs=N_JOBS),
        param_distributions=PARAM_SPACE, n_iter=N_ITER_RF, cv=cv_splits,
        scoring='neg_root_mean_squared_error', n_jobs=N_JOBS,
        random_state=RANDOM_STATE, refit=True)
    search.fit(X_train, y_train)
    cv_rmse = -search.best_score_
    print(f'{bs}m spatial CV -> RMSE={cv_rmse:.3f}  ({time.time() - t0:.1f}s)')
    if cv_rmse < best_rmse_tune:
        best_rmse_tune, best_search, best_block = cv_rmse, search, bs

best_params = best_search.best_params_
print(f'\nBEST: block={best_block}m  CV_RMSE={best_rmse_tune:.3f}')
print(f'params={best_params}')

500m spatial CV -> RMSE=14.777  (105.9s)


1000m spatial CV -> RMSE=14.822  (101.9s)


2000m spatial CV -> RMSE=14.778  (100.7s)

BEST: block=500m  CV_RMSE=14.777
params={'n_estimators': 500, 'min_samples_split': 2, 'min_samples_leaf': 1, 'max_samples': None, 'max_features': 0.5, 'max_depth': None}


---
## Cell 5 · Fit on train, evaluate the holdout, export the predicted raster

In [6]:
rf_fitted = RandomForestRegressor(random_state=RANDOM_STATE, n_jobs=N_JOBS, **best_params)
rf_fitted.fit(X_train, y_train)
joblib.dump(rf_fitted, f'{OUTPUT_DIR}/best_model_RF_split.joblib')

y_pred_test = np.clip(rf_fitted.predict(X_test), 0, 100)
holdout = dict(
    RMSE=float(np.sqrt(mean_squared_error(y_test, y_pred_test))),
    MAE=float(mean_absolute_error(y_test, y_pred_test)),
    R2=float(r2_score(y_test, y_pred_test)),
    Bias=float(np.mean(y_test - y_pred_test)),   # observed - predicted
)
print(f'HOLDOUT (train-only model, scored on the held-out test partition):')
print(f'  RMSE={holdout["RMSE"]:.2f}  MAE={holdout["MAE"]:.2f}  '
      f'R2={holdout["R2"]:.3f}  Bias={holdout["Bias"]:+.2f}')

features = [ee.Feature(None, {**{b: float(row[b]) for b in band_names}, 'IMD': float(row[LABEL_COL])})
            for _, row in train_gdf.iterrows()]
train_fc = ee.FeatureCollection(features)

clf = (ee.Classifier.smileRandomForest(
           numberOfTrees=best_params.get('n_estimators', 200),
           minLeafPopulation=best_params.get('min_samples_leaf', 1),
           seed=RANDOM_STATE)
       .setOutputMode('REGRESSION')
       .train(features=train_fc, classProperty='IMD', inputProperties=band_names))

pred_img = (pred_source_img.classify(clf).rename('IMD_predicted')
            .clip(aoi_geom).updateMask(non_water_mask).clamp(0, 100))

fname = 'IMD_predicted_RF_S2_stackN2_split_Milan'
out_tif = f'{OUTPUT_DIR}/{fname}.tif'
if os.path.exists(out_tif):
    print(f'raster already present at {out_tif} -- skipping export.')
else:
    task = ee.batch.Export.image.toDrive(
        image=pred_img, description=fname, folder=DRIVE_FOLDER,
        fileNamePrefix=fname, region=aoi_geom, scale=10, crs='EPSG:32632',
        maxPixels=1e10, fileFormat='GeoTIFF')
    task.start()
    print(f'submitted batch export task (id={task.id})')
    state = 'READY'
    while state not in ('COMPLETED', 'FAILED', 'CANCELLED'):
        time.sleep(15)
        state = task.status()['state']
        print(f'  {state}')
    if state != 'COMPLETED':
        raise RuntimeError(f'export {state} -- {task.status()}')
    creds = ee.data.get_persistent_credentials()
    drive = gbuild('drive', 'v3', credentials=creds)
    resp = drive.files().list(q=f"name = '{fname}.tif' and trashed = false",
                               fields='files(id, createdTime)').execute()
    files = sorted(resp.get('files', []), key=lambda f: f['createdTime'], reverse=True)
    if not files:
        raise RuntimeError(f'export completed but {fname}.tif not found on Drive.')
    request = drive.files().get_media(fileId=files[0]['id'])
    with open(out_tif, 'wb') as fh:
        downloader = MediaIoBaseDownload(fh, request)
        done = False
        while not done:
            _, done = downloader.next_chunk()
    print(f'downloaded -> {out_tif}')

meta = {
    'band_names': band_names, 'n_train': len(train_gdf), 'n_test': len(test_gdf),
    'dates': STACK_DATES, 'n_dates': len(STACK_DATES),
    'best_block_m': best_block, 'cv_rmse_tune': best_rmse_tune,
    'best_params': best_params, 'holdout_metrics': holdout,
}
with open(f'{OUTPUT_DIR}/model_metadata_split.json', 'w') as f:
    json.dump(meta, f, indent=2, default=str)
print(f'\nSaved -> {OUTPUT_DIR}/model_metadata_split.json')

HOLDOUT (train-only model, scored on the held-out test partition):
  RMSE=13.79  MAE=9.96  R2=0.845  Bias=+0.97


submitted batch export task (id=5IQPKOUMVMIKGSLWTOXOR5X5)


  RUNNING


  RUNNING


  RUNNING


...

  RUNNING


  RUNNING


  RUNNING


  COMPLETED


downloaded -> ./output/milan/clms/stackN2_split/IMD_predicted_RF_S2_stackN2_split_Milan.tif

Saved -> ./output/milan/clms/stackN2_split/model_metadata_split.json
